Open this notebook in a GPU Colab runtime. Configure dataset and checkpoint paths, then run the setup cells and the desired experiment sections. See `docs/TRAINING.md` for inputs and protocol details.


## 1. GPU and Google Drive setup

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No hay GPU activa. Ve a Entorno de ejecución → Cambiar tipo de "
          "entorno de ejecución → GPU, y vuelve a ejecutar esta celda.")

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/resolvfl_cifar100"
os.makedirs(CKPT_DIR, exist_ok=True)
print("Los checkpoints y resultados se guardarán en:", CKPT_DIR)


## 2. Model and training utilities

In [ ]:
%%writefile resolvfl_colab.py
"""
resolvfl_colab.py — ResolvFL para CIFAR-100, listo para Colab.
Con soporte de GPU (device) y checkpointing en Google Drive para poder
parar y retomar entre sesiones sin perder progreso.
"""
from __future__ import annotations
import copy, json, os, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Taxonomía (idéntica a la versión de validación, con certificado)
# ============================================================
class Taxonomy:
    ROOT = "ROOT"

    def __init__(self, parent: Dict[str, str]):
        self.parent = dict(parent)
        names = [self.ROOT] + list(parent.keys())
        self.name2id = {n: i for i, n in enumerate(names)}
        self.id2name = names
        self.children: Dict[int, List[int]] = {self.name2id[n]: [] for n in names}
        for c, p in parent.items():
            self.children[self.name2id[p]].append(self.name2id[c])
        self.leaves: List[int] = [i for i in range(len(names)) if not self.children[i]]
        self.internal: List[int] = [i for i in range(len(names)) if self.children[i]]
        self.leaf_index = {v: k for k, v in enumerate(self.leaves)}
        self.depth = {self.name2id[self.ROOT]: 0}
        self.path: Dict[int, List[int]] = {self.name2id[self.ROOT]: [self.name2id[self.ROOT]]}
        pending = [c for c in parent]
        while pending:
            nxt = []
            for c in pending:
                p = self.name2id[self.parent[c]]
                cid = self.name2id[c]
                if p in self.depth:
                    self.depth[cid] = self.depth[p] + 1
                    self.path[cid] = self.path[p] + [cid]
                else:
                    nxt.append(c)
            pending = nxt
        self.max_depth = max(self.depth.values())
        self.child_slot = {}
        for u in self.internal:
            for slot, c in enumerate(self.children[u]):
                self.child_slot[c] = slot
        self.leaves_under: Dict[int, List[int]] = {}
        for nid in range(len(names)):
            self.leaves_under[nid] = self._collect_leaves(nid)

    def _collect_leaves(self, nid: int) -> List[int]:
        if not self.children[nid]:
            return [nid]
        out = []
        for c in self.children[nid]:
            out += self._collect_leaves(c)
        return out

    def ancestor_at_depth(self, leaf_id: int, d: int) -> int:
        p = self.path[leaf_id]
        return p[min(d, len(p) - 1)]

    def path_edges(self, node_id: int) -> List[Tuple[int, int]]:
        p = self.path[node_id]
        return [(p[j], self.child_slot[p[j + 1]]) for j in range(len(p) - 1)]

    def certificate(self, histograms: List[Dict[int, float]],
                    weights: Optional[List[float]] = None) -> Dict:
        K = len(histograms)
        w = weights or [1.0 / K] * K
        agg = {}
        for wk, h in zip(w, histograms):
            for v, p in h.items():
                agg[v] = agg.get(v, 0.0) + wk * p
        m = {u: 0.0 for u in self.internal}
        order = sorted(range(len(self.id2name)), key=lambda n: -self.depth[n])
        below = {n: 0.0 for n in range(len(self.id2name))}
        for n in order:
            s = 0.0
            for c in self.children[n]:
                s += below[c] + agg.get(c, 0.0)
            below[n] = s
            if n in m:
                m[n] = s
        resolved = {u: m[u] > 0 for u in self.internal}
        maximal_unresolved = [u for u in self.internal
                              if not resolved[u]
                              and all(resolved.get(a, True)
                                      for a in self.path[u][:-1] if a in resolved)]
        blocked = set()
        for u in maximal_unresolved:
            blocked.update(self.leaves_under[u])
        partition = [[l] for l in self.leaves if l not in blocked]
        partition += [list(self.leaves_under[u]) for u in maximal_unresolved]
        return dict(mass=m, resolved=resolved,
                    maximal_unresolved=maximal_unresolved,
                    leaf_partition=partition)


COARSE_TO_FINE = {
    "aquatic_mammals": ["beaver", "dolphin", "otter", "seal", "whale"],
    "fish": ["aquarium_fish", "flatfish", "ray", "shark", "trout"],
    "flowers": ["orchid", "poppy", "rose", "sunflower", "tulip"],
    "food_containers": ["bottle", "bowl", "can", "cup", "plate"],
    "fruit_and_vegetables": ["apple", "mushroom", "orange", "pear", "sweet_pepper"],
    "household_electrical_devices": ["clock", "keyboard", "lamp", "telephone", "television"],
    "household_furniture": ["bed", "chair", "couch", "table", "wardrobe"],
    "insects": ["bee", "beetle", "butterfly", "caterpillar", "cockroach"],
    "large_carnivores": ["bear", "leopard", "lion", "tiger", "wolf"],
    "large_man-made_outdoor_things": ["bridge", "castle", "house", "road", "skyscraper"],
    "large_natural_outdoor_scenes": ["cloud", "forest", "mountain", "plain", "sea"],
    "large_omnivores_and_herbivores": ["camel", "cattle", "chimpanzee", "elephant", "kangaroo"],
    "medium_mammals": ["fox", "porcupine", "possum", "raccoon", "skunk"],
    "non-insect_invertebrates": ["crab", "lobster", "snail", "spider", "worm"],
    "people": ["baby", "boy", "girl", "man", "woman"],
    "reptiles": ["crocodile", "dinosaur", "lizard", "snake", "turtle"],
    "small_mammals": ["hamster", "mouse", "rabbit", "shrew", "squirrel"],
    "trees": ["maple_tree", "oak_tree", "palm_tree", "pine_tree", "willow_tree"],
    "vehicles_1": ["bicycle", "bus", "motorcycle", "pickup_truck", "train"],
    "vehicles_2": ["lawn_mower", "rocket", "streetcar", "tank", "tractor"],
}

def build_cifar100_taxonomy() -> Taxonomy:
    parent = {}
    for coarse, fines in COARSE_TO_FINE.items():
        parent[coarse] = Taxonomy.ROOT
        for f in fines:
            parent[f] = coarse
    return Taxonomy(parent)


# ============================================================
# Modelos
# ============================================================
class ConvBackbone(nn.Module):
    def __init__(self, feat_dim: int = 128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, feat_dim)

    def forward(self, x):
        z = self.conv(x).flatten(1)
        return self.fc(z)


class TreeHead(nn.Module):
    def __init__(self, tax: Taxonomy, feat_dim: int):
        super().__init__()
        self.tax = tax
        self.heads = nn.ModuleDict({
            str(u): nn.Linear(feat_dim, len(tax.children[u])) for u in tax.internal
        })

    def node_logits(self, z, u: int):
        return self.heads[str(u)](z)

    def leaf_log_probs(self, z) -> torch.Tensor:
        logp_node = {self.tax.name2id[Taxonomy.ROOT]: torch.zeros(z.shape[0], device=z.device)}
        order = sorted(self.tax.internal, key=lambda u: self.tax.depth[u])
        for u in order:
            lp = F.log_softmax(self.node_logits(z, u), dim=1)
            for slot, c in enumerate(self.tax.children[u]):
                logp_node[c] = logp_node[u] + lp[:, slot]
        return torch.stack([logp_node[l] for l in self.tax.leaves], dim=1)


class FlatHead(nn.Module):
    def __init__(self, tax: Taxonomy, feat_dim: int):
        super().__init__()
        self.tax = tax
        self.nodes = [n for n in range(len(tax.id2name)) if n != tax.name2id[Taxonomy.ROOT]]
        self.node_pos = {n: i for i, n in enumerate(self.nodes)}
        self.fc = nn.Linear(feat_dim, len(self.nodes))

    def forward(self, z):
        return self.fc(z)


# ============================================================
# Pérdida y agregación (device-aware)
# ============================================================
def marginalized_loss(tax: Taxonomy, backbone, head, x, y_nodes) -> torch.Tensor:
    z = backbone(x)
    device = x.device
    loss = x.new_zeros(())
    n = x.shape[0]
    groups: Dict[int, Tuple[List[int], List[int]]] = {}
    for i, yn in enumerate(y_nodes.tolist()):
        for u, slot in tax.path_edges(yn):
            groups.setdefault(u, ([], []))
            groups[u][0].append(i)
            groups[u][1].append(slot)
    for u, (idx, slots) in groups.items():
        logits = head.node_logits(z[idx], u)
        target = torch.tensor(slots, device=device)
        loss = loss + F.cross_entropy(logits, target, reduction="sum")
    return loss / n


def local_supervision_mass(tax: Taxonomy, y_nodes: np.ndarray) -> Dict[int, float]:
    n = len(y_nodes)
    m = {u: 0 for u in tax.internal}
    for yn in y_nodes:
        for u, _ in tax.path_edges(int(yn)):
            m[u] += 1
    return {u: c / n for u, c in m.items()}


def masked_aggregate(tax, client_states, n_samples, masses):
    total = float(sum(n_samples))
    w = [nk / total for nk in n_samples]
    bb = {key: sum(w[k] * client_states[k][0][key] for k in range(len(w)))
          for key in client_states[0][0]}
    hd = copy.deepcopy(client_states[0][1])
    for u in tax.internal:
        cohort = [k for k in range(len(w)) if masses[k].get(u, 0.0) > 0]
        if not cohort:
            continue
        omega = np.array([w[k] * masses[k][u] for k in cohort])
        omega = omega / omega.sum()
        for suffix in ("weight", "bias"):
            key = f"heads.{u}.{suffix}"
            hd[key] = sum(om * client_states[k][1][key] for om, k in zip(omega, cohort))
    return bb, hd


def plain_average(states, n_samples):
    total = float(sum(n_samples))
    w = [nk / total for nk in n_samples]
    return {key: sum(w[k] * states[k][key] for k in range(len(w))) for key in states[0]}


# ============================================================
# Partición double-Dirichlet
# ============================================================
def double_dirichlet_partition(tax, X, y_leaf, K, alpha_cls, alpha_dep, rng,
                               depth_override=None):
    n_leaves = len(tax.leaves)
    props = rng.dirichlet([alpha_cls] * K, size=n_leaves)
    client_idx = [[] for _ in range(K)]
    for li, leaf in enumerate(tax.leaves):
        idx = np.where(y_leaf == tax.leaf_index[leaf])[0]
        rng.shuffle(idx)
        cuts = (np.cumsum(props[li]) * len(idx)).astype(int)[:-1]
        for k, part in enumerate(np.split(idx, cuts)):
            client_idx[k].extend(part.tolist())
    clients = []
    for k in range(K):
        idx = np.array(sorted(client_idx[k]))
        nu = (depth_override or {}).get(k)
        if nu is None:
            nu = rng.dirichlet([alpha_dep] * tax.max_depth)
        depths = rng.choice(np.arange(1, tax.max_depth + 1), size=len(idx), p=nu)
        y_nodes = np.array([
            tax.ancestor_at_depth(tax.leaves[y_leaf[i]], d)
            for i, d in zip(idx, depths)
        ])
        clients.append(dict(X=X[idx], y_node=y_nodes, y_leaf=y_leaf[idx], nu=nu))
    return clients


# ============================================================
# Checkpointing en Drive (para poder parar y retomar)
# ============================================================
def save_checkpoint(path, round_num, backbone, head, extra=None):
    torch.save({
        "round": round_num,
        "backbone": backbone.state_dict(),
        "head": head.state_dict(),
        "extra": extra or {},
    }, path)


def load_checkpoint(path, backbone, head, map_location="cpu"):
    if not os.path.exists(path):
        return 0, {}
    ckpt = torch.load(path, map_location=map_location)
    backbone.load_state_dict(ckpt["backbone"])
    head.load_state_dict(ckpt["head"])
    return ckpt["round"], ckpt.get("extra", {})


# ============================================================
# Entrenamiento federado, con GPU y checkpoint/resume
# ============================================================
@dataclass
class FedConfig:
    rounds: int = 100
    local_epochs: int = 1
    batch_size: int = 128
    lr: float = 0.05
    feat_dim: int = 128
    seed: int = 0
    save_every: int = 5


def local_train_tree(tax, backbone, head, data, cfg, device):
    opt = torch.optim.SGD(list(backbone.parameters()) + list(head.parameters()),
                          lr=cfg.lr, momentum=0.9)
    X = torch.tensor(data["X"], device=device)
    y = torch.tensor(data["y_node"], device=device)
    n = len(y)
    last_loss = 0.0
    for _ in range(cfg.local_epochs):
        perm = torch.randperm(n, device=device)
        for s in range(0, n, cfg.batch_size):
            b = perm[s:s + cfg.batch_size]
            opt.zero_grad()
            loss = marginalized_loss(tax, backbone, head, X[b], y[b])
            loss.backward()
            opt.step()
            last_loss = loss.item()
    return last_loss


def run_resolvfl(tax, clients, cfg: FedConfig, device, ckpt_path=None,
                 progress_every=5):
    backbone = ConvBackbone(cfg.feat_dim).to(device)
    head = TreeHead(tax, cfg.feat_dim).to(device)
    masses = [local_supervision_mass(tax, c["y_node"]) for c in clients]
    n_samples = [len(c["y_node"]) for c in clients]

    start_round = 0
    if ckpt_path is not None:
        start_round, _ = load_checkpoint(ckpt_path, backbone, head, map_location=device)
        if start_round > 0:
            print(f"    [checkpoint] retomando ResolvFL desde la ronda {start_round}")

    t0 = time.time()
    for rnd in range(start_round, cfg.rounds):
        states, losses = [], []
        for c in clients:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            l = local_train_tree(tax, bb, hd, c, cfg, device)
            losses.append(l)
            states.append(({k: v.detach().clone() for k, v in bb.state_dict().items()},
                           {k: v.detach().clone() for k, v in hd.state_dict().items()}))
        bb_sd, hd_sd = masked_aggregate(tax, states, n_samples, masses)
        backbone.load_state_dict(bb_sd)
        head.load_state_dict(hd_sd)

        if ckpt_path is not None and (rnd + 1) % cfg.save_every == 0:
            save_checkpoint(ckpt_path, rnd + 1, backbone, head)

        if rnd % progress_every == 0 or rnd == cfg.rounds - 1:
            elapsed = time.time() - t0
            print(f"    [ResolvFL] ronda {rnd+1:3d}/{cfg.rounds} | "
                  f"loss medio={np.mean(losses):.4f} | "
                  f"{elapsed:.0f}s transcurridos")

    if ckpt_path is not None:
        save_checkpoint(ckpt_path, cfg.rounds, backbone, head)
    return backbone, head


def run_flat_fedavg(tax, clients, cfg: FedConfig, device, ckpt_path=None,
                    progress_every=5):
    backbone = ConvBackbone(cfg.feat_dim).to(device)
    head = FlatHead(tax, cfg.feat_dim).to(device)
    n_samples = [len(c["y_node"]) for c in clients]

    start_round = 0
    if ckpt_path is not None:
        start_round, _ = load_checkpoint(ckpt_path, backbone, head, map_location=device)
        if start_round > 0:
            print(f"    [checkpoint] retomando FedAvg-flat desde la ronda {start_round}")

    t0 = time.time()
    for rnd in range(start_round, cfg.rounds):
        bb_states, hd_states, losses = [], [], []
        for c in clients:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            opt = torch.optim.SGD(list(bb.parameters()) + list(hd.parameters()),
                                  lr=cfg.lr, momentum=0.9)
            X = torch.tensor(c["X"], device=device)
            y = torch.tensor([hd.node_pos[int(v)] for v in c["y_node"]], device=device)
            n = len(y)
            last_loss = 0.0
            for _ in range(cfg.local_epochs):
                perm = torch.randperm(n, device=device)
                for s in range(0, n, cfg.batch_size):
                    b = perm[s:s + cfg.batch_size]
                    opt.zero_grad()
                    loss = F.cross_entropy(hd(bb(X[b])), y[b])
                    loss.backward()
                    opt.step()
                    last_loss = loss.item()
            losses.append(last_loss)
            bb_states.append({k: v.detach().clone() for k, v in bb.state_dict().items()})
            hd_states.append({k: v.detach().clone() for k, v in hd.state_dict().items()})
        backbone.load_state_dict(plain_average(bb_states, n_samples))
        head.load_state_dict(plain_average(hd_states, n_samples))

        if ckpt_path is not None and (rnd + 1) % cfg.save_every == 0:
            save_checkpoint(ckpt_path, rnd + 1, backbone, head)

        if rnd % progress_every == 0 or rnd == cfg.rounds - 1:
            elapsed = time.time() - t0
            print(f"    [FedAvg-flat] ronda {rnd+1:3d}/{cfg.rounds} | "
                  f"loss medio={np.mean(losses):.4f} | "
                  f"{elapsed:.0f}s transcurridos")

    if ckpt_path is not None:
        save_checkpoint(ckpt_path, cfg.rounds, backbone, head)
    return backbone, head


# ============================================================
# Evaluación
# ============================================================
@torch.no_grad()
def eval_tree(tax, backbone, head, X, y_leaf, device):
    z = backbone(torch.tensor(X, device=device))
    lp = head.leaf_log_probs(z).cpu().numpy()
    pred = lp.argmax(1)
    acc = float((pred == y_leaf).mean())
    return dict(leaf_acc=acc, pred=pred)


@torch.no_grad()
def eval_flat(tax, backbone, head, X, y_leaf, device):
    z = backbone(torch.tensor(X, device=device))
    logits = head(z)
    p = F.softmax(logits, dim=1).cpu().numpy()
    node_pred = p.argmax(1)
    leaf_positions = [head.node_pos[l] for l in tax.leaves]
    abstain = float(np.mean([head.nodes[i] not in tax.leaves for i in node_pred]))
    pred = p[:, leaf_positions].argmax(1)
    acc = float((pred == y_leaf).mean())
    return dict(leaf_acc=acc, abstain_rate=abstain, pred=pred)


# ============================================================
# FedAvg-tree: cabeza en árbol + pérdida marginalizada, pero SIN máscara
# (agregación estándar de todos los clientes en cada nodo). Aísla si la
# ganancia de ResolvFL viene de la máscara o solo de usar la taxonomía.
# ============================================================
def run_fedavg_tree(tax, clients, cfg: FedConfig, device, ckpt_path=None,
                    progress_every=5):
    backbone = ConvBackbone(cfg.feat_dim).to(device)
    head = TreeHead(tax, cfg.feat_dim).to(device)
    n_samples = [len(c["y_node"]) for c in clients]

    start_round = 0
    if ckpt_path is not None:
        start_round, _ = load_checkpoint(ckpt_path, backbone, head, map_location=device)
        if start_round > 0:
            print(f"    [checkpoint] retomando FedAvg-tree desde la ronda {start_round}")

    t0 = time.time()
    for rnd in range(start_round, cfg.rounds):
        bb_list, hd_list, losses = [], [], []
        for c in clients:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            l = local_train_tree(tax, bb, hd, c, cfg, device)
            losses.append(l)
            bb_list.append({k: v.detach().clone() for k, v in bb.state_dict().items()})
            hd_list.append({k: v.detach().clone() for k, v in hd.state_dict().items()})
        backbone.load_state_dict(plain_average(bb_list, n_samples))
        head.load_state_dict(plain_average(hd_list, n_samples))  # UNMASKED: every node averaged over all clients

        if ckpt_path is not None and (rnd + 1) % cfg.save_every == 0:
            save_checkpoint(ckpt_path, rnd + 1, backbone, head)
        if rnd % progress_every == 0 or rnd == cfg.rounds - 1:
            elapsed = time.time() - t0
            print(f"    [FedAvg-tree] ronda {rnd+1:3d}/{cfg.rounds} | "
                  f"loss medio={np.mean(losses):.4f} | {elapsed:.0f}s transcurridos")

    if ckpt_path is not None:
        save_checkpoint(ckpt_path, cfg.rounds, backbone, head)
    return backbone, head


## 3. CIFAR-100 and federated partition

In [ ]:
import numpy as np
import torch

from resolvfl_colab import (build_cifar100_taxonomy, double_dirichlet_partition,
                            FedConfig, run_resolvfl, run_flat_fedavg,
                            eval_tree, eval_flat, run_fedavg_tree)

tax = build_cifar100_taxonomy()
print(f"Taxonomía: {len(tax.leaves)} hojas, {len(tax.internal)-1} superclases (+ROOT), "
      f"profundidad {tax.max_depth}")
assert len(tax.leaves) == 100 and len(tax.internal) == 21, "algo no cuadra con CIFAR-100"

# Nota: usamos el mirror de Hugging Face en vez del servidor oficial de
# Toronto (cs.toronto.edu), que a fecha de este notebook está teniendo
# problemas de velocidad reportados por muchos usuarios de torchvision
# (ver https://github.com/pytorch/vision/issues/9536). Este mirror es
# estable y no depende de ese servidor.
print("Instalando 'datasets' de Hugging Face...")
%pip install -q datasets

from datasets import load_dataset
print("Descargando CIFAR-100 desde Hugging Face (solo la primera vez)...")
hf_ds = load_dataset("uoft-cs/cifar100")
fine_names = hf_ds["train"].features["fine_label"].names

# El propio dataset en HF tiene un typo conocido en sus metadatos: la clase
# 26 aparece como "cra" en vez de "crab". Buscamos por coincidencia exacta
# y, si falla, por prefijo único (cubre ese caso y cualquier otro similar
# sin tener que parchear el nombre a mano).
def resolve_name(n):
    if n in tax.name2id:
        return tax.name2id[n]
    candidates = [full for full in tax.name2id if full.startswith(n)]
    if len(candidates) == 1:
        return tax.name2id[candidates[0]]
    raise KeyError(f"No se pudo resolver el nombre de clase '{n}' contra la taxonomía")

leaf_id_of_fine = [resolve_name(n) for n in fine_names]

def pack(split):
    X = np.stack([np.array(img) for img in split["img"]]).astype(np.float32) / 255.0
    X = X.transpose(0, 3, 1, 2)
    y_fine = np.array(split["fine_label"])
    y_leaf = np.array([tax.leaf_index[leaf_id_of_fine[f]] for f in y_fine])
    return X, y_leaf

Xtr, ytr = pack(hf_ds["train"])
Xte, yte = pack(hf_ds["test"])
print(f"Listo: train={Xtr.shape}, test={Xte.shape}")


In [ ]:
K = 10
rng = np.random.default_rng(1)
clients = double_dirichlet_partition(tax, Xtr, ytr, K,
                                     alpha_cls=0.5, alpha_dep=0.5, rng=rng)
for k, c in enumerate(clients):
    print(f"  cliente {k}: {len(c['y_node'])} muestras | "
          f"perfil de profundidad = {np.round(c['nu'], 2)}")

hists = []
for c in clients:
    vals, cnts = np.unique(c["y_node"], return_counts=True)
    hists.append({int(v): cnt / len(c["y_node"]) for v, cnt in zip(vals, cnts)})
cert = tax.certificate(hists)
n_unresolved = sum(1 for v in cert["resolved"].values() if not v)
print(f"\nCertificado: {n_unresolved} de {len(cert['resolved'])} superclases sin resolver "
      f"con esta partición.")


## 4. ResolvFL training

Checkpoints are saved every five rounds; an existing compatible checkpoint resumes from its saved round.

In [ ]:
cfg = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                feat_dim=128, seed=1, save_every=5)

ckpt_resolvfl = os.path.join(CKPT_DIR, "resolvfl.pt")
backbone_fg, head_fg = run_resolvfl(tax, clients, cfg, device, ckpt_path=ckpt_resolvfl)

res_fg = eval_tree(tax, backbone_fg, head_fg, Xte, yte, device)
print(f"\n>>> ResolvFL — accuracy en hojas (test): {res_fg['leaf_acc']:.4f}")


## 5. Flat baseline

Flat classification over observed taxonomy nodes with checkpoint support.

In [ ]:
ckpt_flat = os.path.join(CKPT_DIR, "flat_fedavg.pt")
backbone_fl, head_fl = run_flat_fedavg(tax, clients, cfg, device, ckpt_path=ckpt_flat)

res_fl = eval_flat(tax, backbone_fl, head_fl, Xte, yte, device)
print(f"\n>>> FedAvg-flat — accuracy en hojas (test): {res_fl['leaf_acc']:.4f}")
print(f">>> FedAvg-flat — tasa de abstención a nodo interno: {res_fl['abstain_rate']:.4f}")


## 5b. Tree baseline

Tree uses the same hierarchical architecture and path loss as ResolvFL, with all-client aggregation for each node.

In [ ]:
ckpt_tree = os.path.join(CKPT_DIR, "fedavg_tree.pt")
backbone_tree, head_tree = run_fedavg_tree(tax, clients, cfg, device, ckpt_path=ckpt_tree)

res_tree = eval_tree(tax, backbone_tree, head_tree, Xte, yte, device)
print(f"\n>>> FedAvg-tree (sin máscara) — accuracy en hojas (test): {res_tree['leaf_acc']:.4f}")


## 6. Save result summaries

In [ ]:
import json

# Cargamos lo que ya hubiera en Drive de sesiones anteriores y AÑADIMOS a
# eso, en vez de reemplazarlo — así ninguna sección posterior (8, 9, 10,
# 11...) pierde su progreso si esta celda se vuelve a ejecutar en una
# sesión nueva.
_summary_path = os.path.join(CKPT_DIR, "results_summary.json")
summary = {}
if os.path.exists(_summary_path):
    with open(_summary_path) as f:
        summary = json.load(f)

summary.update({
    "resolvfl_leaf_acc": res_fg["leaf_acc"],
    "flat_fedavg_leaf_acc": res_fl["leaf_acc"],
    "flat_fedavg_abstain_rate": res_fl["abstain_rate"],
    "fedavg_tree_leaf_acc": res_tree["leaf_acc"],
    "rounds": cfg.rounds,
    "num_clients": K,
    "alpha_cls": 0.5,
    "alpha_dep": 0.5,
})
with open(_summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Resumen guardado en Drive:", _summary_path)
print()
for k, v in summary.items():
    print(f"  {k}: {v}")


## 7. Coarse-only and fine-only clients

This exploratory setting uses five superclass-only clients and five leaf-label clients. Only the latter directly supervise the superclass heads.

In [ ]:
K_half = K // 2
override = {k: np.array([1.0, 0.0]) for k in range(K_half)}       # solo grueso
override.update({k: np.array([0.0, 1.0]) for k in range(K_half, K)})  # solo fino

rng2 = np.random.default_rng(2)
clients_extreme = double_dirichlet_partition(tax, Xtr, ytr, K,
                                              alpha_cls=0.5, alpha_dep=0.5,
                                              rng=rng2, depth_override=override)

hists_extreme = []
for c in clients_extreme:
    vals, cnts = np.unique(c["y_node"], return_counts=True)
    hists_extreme.append({int(v): cnt / len(c["y_node"]) for v, cnt in zip(vals, cnts)})
cert_extreme = tax.certificate(hists_extreme)
n_unresolved = sum(1 for v in cert_extreme["resolved"].values() if not v)
print(f"Superclases sin resolver bajo heterogeneidad extrema: "
      f"{n_unresolved}/{len(cert_extreme['resolved'])}")
print("(Puede seguir dando 0: con reparto de clases moderado, basta con que "
      "algún cliente fino tenga aunque sea una muestra de cada superclase "
      "para que quede 'resuelta' — lo que cambia de verdad aquí es el "
      "TAMAÑO del cohorte que entrena cada cabeza, no si está resuelta.)")


In [ ]:
ckpt_fg_extreme = os.path.join(CKPT_DIR, "resolvfl_extreme.pt")
backbone_fg_x, head_fg_x = run_resolvfl(tax, clients_extreme, cfg, device,
                                        ckpt_path=ckpt_fg_extreme)
res_fg_x = eval_tree(tax, backbone_fg_x, head_fg_x, Xte, yte, device)
print(f"\n>>> ResolvFL (masked), heterogeneidad extrema: "
      f"{res_fg_x['leaf_acc']:.4f}")

ckpt_tree_extreme = os.path.join(CKPT_DIR, "fedavg_tree_extreme.pt")
backbone_tree_x, head_tree_x = run_fedavg_tree(tax, clients_extreme, cfg, device,
                                               ckpt_path=ckpt_tree_extreme)
res_tree_x = eval_tree(tax, backbone_tree_x, head_tree_x, Xte, yte, device)
print(f">>> FedAvg-tree (unmasked), heterogeneidad extrema: "
      f"{res_tree_x['leaf_acc']:.4f}")

print(f"\nDiferencia (masked - unmasked): "
      f"{res_fg_x['leaf_acc'] - res_tree_x['leaf_acc']:+.4f}")


In [ ]:
summary["resolvfl_extreme_leaf_acc"] = res_fg_x["leaf_acc"]
summary["fedavg_tree_extreme_leaf_acc"] = res_tree_x["leaf_acc"]
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Resumen actualizado en Drive con la ablación extrema.")
for k, v in summary.items():
    print(f"  {k}: {v}")


## 8. Short-run training trajectories

Compare Tree and ResolvFL for 30 rounds on the same coarse/fine partition, evaluating every ten rounds. These exploratory trajectories do not establish a convergence rate.

In [ ]:
# Salto inteligente: si ya corrió antes (está en el resumen de Drive),
# no lo repite.
_summary_path = os.path.join(CKPT_DIR, "results_summary.json")
_prev_summary = {}
if os.path.exists(_summary_path):
    with open(_summary_path) as f:
        _prev_summary = json.load(f)

if "trajectory_masked_30r" in _prev_summary:
    print("Ya hecho antes (encontrado en Drive) — no se repite.")
    traj_masked = _prev_summary["trajectory_masked_30r"]
    traj_unmasked = _prev_summary["trajectory_unmasked_30r"]
    summary.update(_prev_summary)
    print("--- Trayectorias (cargadas de Drive) ---")
    for (r1, a1), (r2, a2) in zip(traj_masked, traj_unmasked):
        print(f"  ronda {r1:2d}: {a1:.4f} vs {a2:.4f}  (dif: {a1-a2:+.4f})")
else:
    # Config corta: 30 rondas, evaluando la trayectoria
    from resolvfl_colab import ConvBackbone, TreeHead, local_train_tree, \
        masked_aggregate, plain_average, local_supervision_mass
    import copy, time

    cfg_short = FedConfig(rounds=30, local_epochs=1, batch_size=128, lr=0.05,
                          feat_dim=128, seed=3, save_every=10)

    def run_with_trajectory(masked: bool, label: str):
        torch.manual_seed(cfg_short.seed)
        backbone = ConvBackbone(cfg_short.feat_dim).to(device)
        head = TreeHead(tax, cfg_short.feat_dim).to(device)
        masses = [local_supervision_mass(tax, c["y_node"]) for c in clients_extreme]
        n_samples = [len(c["y_node"]) for c in clients_extreme]
        trajectory = []
        t0 = time.time()
        for rnd in range(cfg_short.rounds):
            bb_list, hd_list = [], []
            for c in clients_extreme:
                bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
                local_train_tree(tax, bb, hd, c, cfg_short, device)
                bb_list.append({k: v.detach().clone() for k, v in bb.state_dict().items()})
                hd_list.append({k: v.detach().clone() for k, v in hd.state_dict().items()})
            if masked:
                bb_sd, hd_sd = masked_aggregate(tax, list(zip(bb_list, hd_list)),
                                                n_samples, masses)
                backbone.load_state_dict(bb_sd); head.load_state_dict(hd_sd)
            else:
                backbone.load_state_dict(plain_average(bb_list, n_samples))
                head.load_state_dict(plain_average(hd_list, n_samples))
            if (rnd + 1) % 10 == 0 or rnd == cfg_short.rounds - 1:
                acc = eval_tree(tax, backbone, head, Xte, yte, device)["leaf_acc"]
                trajectory.append((rnd + 1, acc))
                print(f"    [{label}] ronda {rnd+1:2d}/{cfg_short.rounds} | "
                      f"acc test = {acc:.4f} | {time.time()-t0:.0f}s")
        return trajectory

    print("=== ResolvFL (masked), 30 rondas ===")
    traj_masked = run_with_trajectory(masked=True, label="masked")
    print("\n=== FedAvg-tree (unmasked), 30 rondas ===")
    traj_unmasked = run_with_trajectory(masked=False, label="unmasked")

    print("\n--- Trayectorias comparadas (ronda: masked vs unmasked) ---")
    for (r1, a1), (r2, a2) in zip(traj_masked, traj_unmasked):
        print(f"  ronda {r1:2d}: {a1:.4f} vs {a2:.4f}  (dif: {a1-a2:+.4f})")

    summary["trajectory_masked_30r"] = traj_masked
    summary["trajectory_unmasked_30r"] = traj_unmasked
    with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print("\nGuardado en Drive.")


## 9. Node-wise update-rescaling control

Compare active-cohort aggregation, all-client aggregation, and all-client aggregation with rescaled head updates. The exploratory comparison uses 30 rounds and three seeds.

In [ ]:
from resolvfl_colab import ConvBackbone, TreeHead, local_train_tree, \
    masked_aggregate, plain_average, local_supervision_mass
import copy, time, json

def aggregate_rescaled(tax, prev_head_sd, bb_list, hd_list, n_samples, masses):
    """Promedio sin máscara + amplificación del delta de cada cabeza por
    1/W_S(u), donde W_S(u) es la fracción de peso de los clientes que
    supervisan u. Si nadie supervisa u, la cabeza no se toca."""
    bb_avg = plain_average(bb_list, n_samples)
    hd_avg = plain_average(hd_list, n_samples)
    total = float(sum(n_samples))
    w = [nk / total for nk in n_samples]
    hd_new = copy.deepcopy(hd_avg)
    for u in tax.internal:
        W_S = sum(w[k] for k in range(len(w)) if masses[k].get(u, 0.0) > 0)
        for suffix in ("weight", "bias"):
            key = f"heads.{u}.{suffix}"
            if W_S <= 0:
                hd_new[key] = prev_head_sd[key].clone()
            else:
                delta = hd_avg[key] - prev_head_sd[key]
                hd_new[key] = prev_head_sd[key] + delta / W_S
    return bb_avg, hd_new

def run_trajectory(mode: str, seed: int, rounds: int = 30):
    """mode: \'masked\' | \'unmasked\' | \'rescaled\'"""
    torch.manual_seed(seed)
    cfg_s = FedConfig(rounds=rounds, local_epochs=1, batch_size=128, lr=0.05,
                      feat_dim=128, seed=seed)
    backbone = ConvBackbone(cfg_s.feat_dim).to(device)
    head = TreeHead(tax, cfg_s.feat_dim).to(device)
    masses = [local_supervision_mass(tax, c["y_node"]) for c in clients_extreme]
    n_samples = [len(c["y_node"]) for c in clients_extreme]
    traj = []
    for rnd in range(rounds):
        prev_head_sd = {k: v.detach().clone() for k, v in head.state_dict().items()}
        bb_list, hd_list = [], []
        for c in clients_extreme:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            local_train_tree(tax, bb, hd, c, cfg_s, device)
            bb_list.append({k: v.detach().clone() for k, v in bb.state_dict().items()})
            hd_list.append({k: v.detach().clone() for k, v in hd.state_dict().items()})
        if mode == "masked":
            bb_sd, hd_sd = masked_aggregate(tax, list(zip(bb_list, hd_list)),
                                            n_samples, masses)
        elif mode == "rescaled":
            bb_sd, hd_sd = aggregate_rescaled(tax, prev_head_sd, bb_list, hd_list,
                                              n_samples, masses)
        else:
            bb_sd, hd_sd = plain_average(bb_list, n_samples), plain_average(hd_list, n_samples)
        backbone.load_state_dict(bb_sd)
        head.load_state_dict(hd_sd)
        if (rnd + 1) % 10 == 0:
            acc = eval_tree(tax, backbone, head, Xte, yte, device)["leaf_acc"]
            traj.append(acc)
    return traj  # [acc@10, acc@20, acc@30]

# ------------------------------------------------------------------
# Guardado incremental: cada (semilla, modo) se guarda en cuanto termina,
# en un JSON aparte en Drive. Si Colab se corta a mitad, al re-ejecutar
# esta celda solo se hacen los que faltan, no se repite nada.
# ------------------------------------------------------------------
GRID_PATH = os.path.join(CKPT_DIR, "grid_3seed.json")
SEEDS = [3, 4, 5]
MODES = ["masked", "unmasked", "rescaled"]

grid = {}
if os.path.exists(GRID_PATH):
    with open(GRID_PATH) as f:
        grid = json.load(f)
    print(f"Progreso previo encontrado en Drive: {len(grid)} combinaciones ya hechas.")

t0 = time.time()
for seed in SEEDS:
    for mode in MODES:
        key = f"{mode}_seed{seed}"
        if key in grid:
            print(f"  [ya hecho] {key}: {grid[key]}")
            continue
        traj = run_trajectory(mode, seed)
        grid[key] = traj
        with open(GRID_PATH, "w") as f:
            json.dump(grid, f, indent=2)
        print(f"  seed {seed} | {mode:9s} | acc@10/20/30 = "
              f"{traj[0]:.4f} / {traj[1]:.4f} / {traj[2]:.4f} "
              f"| {time.time()-t0:.0f}s total | guardado en Drive")

print("\n--- Media ± desviación (3 semillas) ---")
import numpy as np
results = {m: [grid[f"{m}_seed{s}"] for s in SEEDS] for m in MODES}
for mode, runs in results.items():
    arr = np.array(runs)  # (seeds, 3)
    means, stds = arr.mean(0), arr.std(0)
    print(f"  {mode:9s}: " + "  ".join(
        f"R{r}: {m_:.4f}±{s_:.4f}" for r, m_, s_ in zip([10, 20, 30], means, stds)))

summary["trajectories_3seed"] = results
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nResumen final guardado en Drive.")


## 10. Coverage-removal intervention: vehicles_1

Replace descendant labels with the superclass label for all examples in `vehicles_1`. Audit observed path-supervision counts before training and compare target and outside leaf accuracy. Ancestor-of-decoded-leaf diagnostics in this source are not marginal-prefix accuracy. The intervention provides empirical evidence; it does not prove the population theorem.

In [ ]:
VIOLATED = "vehicles_1"
vid = tax.name2id[VIOLATED]
under = set(tax.leaves_under[vid])

def censor_below(node_id, clients_list):
    out = []
    for c in clients_list:
        y = c["y_node"].copy()
        for i, yn in enumerate(y):
            if yn in under or yn == node_id:
                y[i] = node_id
        out.append(dict(c, y_node=y))
    return out

clients_violated = censor_below(vid, clients)  # partición moderada original

# El certificado, ANTES de entrenar:
hists_v = []
for c in clients_violated:
    vals, cnts = np.unique(c["y_node"], return_counts=True)
    hists_v.append({int(v): cnt / len(c["y_node"]) for v, cnt in zip(vals, cnts)})
cert_v = tax.certificate(hists_v)
unresolved_names = [tax.id2name[u] for u in cert_v["maximal_unresolved"]]
print("Nodos sin resolver detectados por el certificado:", unresolved_names)
print("Partición de hojas que el certificado declara indistinguibles:")
for blk in cert_v["leaf_partition"]:
    if len(blk) > 1:
        print("   ", [tax.id2name[l] for l in blk])


In [ ]:
cfg_e1 = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                   feat_dim=128, seed=1, save_every=5)
ckpt_e1 = os.path.join(CKPT_DIR, "resolvfl_violated.pt")
backbone_v, head_v = run_resolvfl(tax, clients_violated, cfg_e1, device,
                                  ckpt_path=ckpt_e1)
res_v = eval_tree(tax, backbone_v, head_v, Xte, yte, device)
print(f"\nAccuracy global bajo violación: {res_v['leaf_acc']:.4f}")


In [ ]:
# Comparación quirúrgica contra el run resoluble original (checkpoint ya en Drive)
from resolvfl_colab import ConvBackbone, TreeHead, load_checkpoint

backbone_ref = ConvBackbone(128).to(device)
head_ref = TreeHead(tax, 128).to(device)
_r, _ = load_checkpoint(os.path.join(CKPT_DIR, "resolvfl.pt"),
                        backbone_ref, head_ref, map_location=device)
print(f"Checkpoint resoluble cargado (ronda {_r}).")
res_ref = eval_tree(tax, backbone_ref, head_ref, Xte, yte, device)

violated_leaf_idx = [tax.leaf_index[l] for l in under]
other_leaf_idx = [tax.leaf_index[l] for l in tax.leaves if l not in under]

def block_acc(pred, y, idxs):
    mask = np.isin(y, idxs)
    return float((pred[mask] == y[mask]).mean())

acc_viol_v = block_acc(res_v["pred"], yte, violated_leaf_idx)
acc_viol_ref = block_acc(res_ref["pred"], yte, violated_leaf_idx)
acc_other_v = block_acc(res_v["pred"], yte, other_leaf_idx)
acc_other_ref = block_acc(res_ref["pred"], yte, other_leaf_idx)

mask_viol = np.isin(yte, violated_leaf_idx)
inside = np.isin(res_v["pred"][mask_viol], violated_leaf_idx)
superclass_acc = float(inside.mean())

print(f"\n================ E1: Teorema 1(b) ================")
print(f"Hojas de {VIOLATED} (violadas):")
print(f"   run resoluble: {acc_viol_ref:.4f}  |  run violado: {acc_viol_v:.4f}")
print(f"   (predicción: colapso hacia ~azar dentro de la superclase, ~0.2·acc_superclase)")
print(f"Las otras 95 hojas (resueltas):")
print(f"   run resoluble: {acc_other_ref:.4f}  |  run violado: {acc_other_v:.4f}")
print(f"   (predicción: diferencia ~0 — la degradación NO se propaga)")
print(f"Accuracy a nivel superclase para muestras de {VIOLATED}: {superclass_acc:.4f}")
print(f"   (predicción: alta — el prefijo resuelto del árbol sobrevive)")

summary["E1"] = dict(violated_node=VIOLATED,
                     violated_leaves_resolvable=acc_viol_ref,
                     violated_leaves_violated=acc_viol_v,
                     other_leaves_resolvable=acc_other_ref,
                     other_leaves_violated=acc_other_v,
                     superclass_acc_violated=superclass_acc)
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nGuardado en Drive.")


## 11. Three-seed coverage-removal comparison

Reuse available seed-1 checkpoints and train the remaining paired reference/intervention runs for 150 rounds. Saved outputs support descriptive means, sample standard deviations and paired differences.

In [ ]:
E1_GRID_PATH = os.path.join(CKPT_DIR, "e1_multiseed.json")
E1_SEEDS = [1, 2, 3]

e1_grid = {}
if os.path.exists(E1_GRID_PATH):
    with open(E1_GRID_PATH) as f:
        e1_grid = json.load(f)
    print(f"Progreso previo de E1 multi-semilla encontrado: {len(e1_grid)} semillas ya hechas.")

def compute_e1_metrics(backbone_ref, head_ref, backbone_v, head_v):
    res_ref = eval_tree(tax, backbone_ref, head_ref, Xte, yte, device)
    res_v = eval_tree(tax, backbone_v, head_v, Xte, yte, device)
    acc_viol_ref = block_acc(res_ref["pred"], yte, violated_leaf_idx)
    acc_viol_v = block_acc(res_v["pred"], yte, violated_leaf_idx)
    acc_other_ref = block_acc(res_ref["pred"], yte, other_leaf_idx)
    acc_other_v = block_acc(res_v["pred"], yte, other_leaf_idx)
    mask_viol = np.isin(yte, violated_leaf_idx)
    inside = np.isin(res_v["pred"][mask_viol], violated_leaf_idx)
    superclass_acc = float(inside.mean())
    return dict(violated_resolvable=acc_viol_ref, violated_violated=acc_viol_v,
               other_resolvable=acc_other_ref, other_violated=acc_other_v,
               superclass_acc=superclass_acc)

for seed in E1_SEEDS:
    key = f"seed{seed}"
    if key in e1_grid:
        print(f"  [ya hecho] semilla {seed}: {e1_grid[key]}")
        continue

    cfg_seed = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                         feat_dim=128, seed=seed, save_every=10)

    # Referencia resoluble para esta semilla (reutiliza checkpoint si es la semilla 1)
    ckpt_ref_s = os.path.join(CKPT_DIR, f"resolvfl_seed{seed}.pt" if seed != 1
                              else "resolvfl.pt")
    print(f"\n--- Semilla {seed}: entrenando referencia resoluble ---")
    torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves)
    bb_ref_s, hd_ref_s = run_resolvfl(tax, clients, cfg_seed, device,
                                      ckpt_path=ckpt_ref_s)

    # Modelo bajo violación para esta semilla
    ckpt_v_s = os.path.join(CKPT_DIR, f"resolvfl_violated_seed{seed}.pt" if seed != 1
                            else "resolvfl_violated.pt")
    print(f"--- Semilla {seed}: entrenando bajo violación ---")
    torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves)
    bb_v_s, hd_v_s = run_resolvfl(tax, clients_violated, cfg_seed, device,
                                  ckpt_path=ckpt_v_s)

    metrics = compute_e1_metrics(bb_ref_s, hd_ref_s, bb_v_s, hd_v_s)
    e1_grid[key] = metrics
    with open(E1_GRID_PATH, "w") as f:
        json.dump(e1_grid, f, indent=2)
    print(f"  Semilla {seed} completa y guardada en Drive: {metrics}")

print("\n--- E1: media ± desviación (3 semillas) ---")
keys = ["violated_resolvable", "violated_violated",
       "other_resolvable", "other_violated", "superclass_acc"]
arr = {k: np.array([e1_grid[f"seed{s}"][k] for s in E1_SEEDS]) for k in keys}
for k in keys:
    print(f"  {k:20s}: {arr[k].mean():.4f} ± {arr[k].std():.4f}")

summary["E1_multiseed"] = e1_grid
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nGuardado en Drive.")


## 12. Additional intervention targets

Apply the same coverage-removal protocol to `small_mammals` and `flowers`, with the corresponding reference runs.

In [ ]:
MULTI_NODES = ["small_mammals", "flowers"]
MULTI_GRID_PATH = os.path.join(CKPT_DIR, "multi_node_violation.json")

multi_grid = {}
if os.path.exists(MULTI_GRID_PATH):
    with open(MULTI_GRID_PATH) as f:
        multi_grid = json.load(f)
    print(f"Progreso previo encontrado: {list(multi_grid.keys())}")

def censor_below_node(node_name, clients_list, tax):
    """Versión autocontenida: no depende de ninguna variable global,
    cada nodo calcula su propio conjunto de hojas a censurar."""
    node_id = tax.name2id[node_name]
    under_node = set(tax.leaves_under[node_id])
    out = []
    for c in clients_list:
        y = c["y_node"].copy()
        for i, yn in enumerate(y):
            if yn in under_node or yn == node_id:
                y[i] = node_id
        out.append(dict(c, y_node=y))
    return out, under_node

# Referencia resoluble: la misma para los tres nodos (ya entrenada)
backbone_ref_m = ConvBackbone(128).to(device)
head_ref_m = TreeHead(tax, 128).to(device)
_r, _ = load_checkpoint(os.path.join(CKPT_DIR, "resolvfl.pt"),
                        backbone_ref_m, head_ref_m, map_location=device)
res_ref_m = eval_tree(tax, backbone_ref_m, head_ref_m, Xte, yte, device)
print(f"Referencia resoluble cargada (ronda {_r}).")

cfg_m = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                  feat_dim=128, seed=1, save_every=10)

for node_name in MULTI_NODES:
    if node_name in multi_grid:
        print(f"\n[ya hecho] {node_name}: {multi_grid[node_name]}")
        continue

    print(f"\n=== Nodo: {node_name} ===")
    clients_node, under_node = censor_below_node(node_name, clients, tax)

    hists_node = []
    for c in clients_node:
        vals, cnts = np.unique(c["y_node"], return_counts=True)
        hists_node.append({int(v): cnt / len(c["y_node"]) for v, cnt in zip(vals, cnts)})
    cert_node = tax.certificate(hists_node)
    unresolved = [tax.id2name[u] for u in cert_node["maximal_unresolved"]]
    print(f"  Certificado (antes de entrenar): sin resolver = {unresolved}")
    assert unresolved == [node_name], f"esperaba solo {node_name}, salió {unresolved}"

    ckpt_node = os.path.join(CKPT_DIR, f"resolvfl_violated_{node_name}.pt")
    bb_node, hd_node = run_resolvfl(tax, clients_node, cfg_m, device, ckpt_path=ckpt_node)
    res_node = eval_tree(tax, bb_node, hd_node, Xte, yte, device)

    violated_idx = [tax.leaf_index[l] for l in under_node]
    other_idx = [tax.leaf_index[l] for l in tax.leaves if l not in under_node]

    acc_viol_ref = block_acc(res_ref_m["pred"], yte, violated_idx)
    acc_viol_v = block_acc(res_node["pred"], yte, violated_idx)
    acc_other_ref = block_acc(res_ref_m["pred"], yte, other_idx)
    acc_other_v = block_acc(res_node["pred"], yte, other_idx)
    mask_v = np.isin(yte, violated_idx)
    superclass_acc = float(np.isin(res_node["pred"][mask_v], violated_idx).mean())

    result = dict(node=node_name,
                 violated_resolvable=acc_viol_ref, violated_violated=acc_viol_v,
                 other_resolvable=acc_other_ref, other_violated=acc_other_v,
                 superclass_acc=superclass_acc)
    multi_grid[node_name] = result
    with open(MULTI_GRID_PATH, "w") as f:
        json.dump(multi_grid, f, indent=2)

    print(f"  Hojas de {node_name}: {acc_viol_ref:.4f} -> {acc_viol_v:.4f}")
    print(f"  Otras 95 hojas:       {acc_other_ref:.4f} -> {acc_other_v:.4f}")
    print(f"  Identificación superclase: {superclass_acc:.4f}")
    print(f"  Guardado en Drive.")

print("\n=== Resumen multi-nodo (incluyendo vehicles_1 ya hecho) ===")
print(f"{'Nodo':16s} {'viol_ref':>9s} {'viol_viol':>9s} {'other_ref':>9s} {'other_viol':>10s} {'superclass':>10s}")
if "E1" in summary:
    e1v = summary["E1"]
    print(f"{'vehicles_1':16s} {e1v['violated_leaves_resolvable']:9.4f} "
         f"{e1v['violated_leaves_violated']:9.4f} {e1v['other_leaves_resolvable']:9.4f} "
         f"{e1v['other_leaves_violated']:10.4f} {e1v['superclass_acc_violated']:10.4f}")
for node_name, r in multi_grid.items():
    print(f"{node_name:16s} {r['violated_resolvable']:9.4f} {r['violated_violated']:9.4f} "
         f"{r['other_resolvable']:9.4f} {r['other_violated']:10.4f} {r['superclass_acc']:10.4f}")

summary["multi_node_violation"] = multi_grid
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nGuardado final en Drive.")


## 13. Multiple targets and seeds

Reuse reference checkpoints across intervention targets. Save target and outside leaf accuracy and the historical decoded-leaf-ancestor diagnostics separately. These diagnostics must not be reported as marginal-prefix accuracy.

In [ ]:
FULL_GRID_PATH = os.path.join(CKPT_DIR, "e1_full_grid.json")
ALL_NODES = ["vehicles_1", "small_mammals", "flowers"]
ALL_SEEDS = [1, 2, 3]

full_grid = {}
if os.path.exists(FULL_GRID_PATH):
    with open(FULL_GRID_PATH) as f:
        full_grid = json.load(f)
    print(f"Progreso previo: {len(full_grid)} combinaciones ya hechas.")

def censor_below_node(node_name, clients_list, tax):
    node_id = tax.name2id[node_name]
    under_node = set(tax.leaves_under[node_id])
    out = []
    for c in clients_list:
        y = c["y_node"].copy()
        for i, yn in enumerate(y):
            if yn in under_node or yn == node_id:
                y[i] = node_id
        out.append(dict(c, y_node=y))
    return out, under_node

def superclass_acc_from_pred(pred, y_true, violated_idx):
    """Fracción de muestras (cuya etiqueta real cae en el bloque violado)
    cuya predicción también cae dentro del bloque, sin importar la hoja
    exacta. Se aplica tanto al modelo resoluble como al violado."""
    mask = np.isin(y_true, violated_idx)
    if mask.sum() == 0:
        return float("nan")
    return float(np.isin(pred[mask], violated_idx).mean())

for node_name in ALL_NODES:
    node_id = tax.name2id[node_name]
    under_node = set(tax.leaves_under[node_id])
    violated_idx = [tax.leaf_index[l] for l in under_node]
    other_idx = [tax.leaf_index[l] for l in tax.leaves if l not in under_node]

    for seed in ALL_SEEDS:
        key = f"{node_name}_seed{seed}"
        if key in full_grid:
            print(f"  [ya hecho] {key}")
            continue

        cfg_s = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                          feat_dim=128, seed=seed, save_every=10)

        # Referencia resoluble: una por semilla, compartida entre los 3 nodos
        # (mismos nombres que ya usó la Seccion 11 -- se reutiliza sin reentrenar)
        ckpt_ref_s = os.path.join(CKPT_DIR, f"resolvfl_seed{seed}.pt" if seed != 1
                                  else "resolvfl.pt")
        print(f"\n--- {node_name}, semilla {seed}: referencia resoluble ---")
        torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves / E1 / overlap sweep)
        bb_ref_s, hd_ref_s = run_resolvfl(tax, clients, cfg_s, device,
                                          ckpt_path=ckpt_ref_s)
        res_ref_s = eval_tree(tax, bb_ref_s, hd_ref_s, Xte, yte, device)

        # Modelo bajo violación de este nodo, esta semilla.
        # Use the exact filenames from the original runs
        # las Secciones 11 y 12, si no, se reentrena por error algo que ya
        # esta hecho:
        #   - vehicles_1, semilla 1  -> "resolvfl_violated.pt"        (Sec. 10/11)
        #   - vehicles_1, semilla 2/3 -> "resolvfl_violated_seed{N}.pt" (Sec. 11)
        #   - small_mammals/flowers, semilla 1 -> "resolvfl_violated_{nodo}.pt" (Sec. 12)
        #   - small_mammals/flowers, semilla 2/3 -> nombre nuevo (no existia antes)
        clients_node, _ = censor_below_node(node_name, clients, tax)
        if node_name == "vehicles_1":
            ckpt_v_name = "resolvfl_violated.pt" if seed == 1 else f"resolvfl_violated_seed{seed}.pt"
        else:
            ckpt_v_name = (f"resolvfl_violated_{node_name}.pt" if seed == 1
                           else f"resolvfl_violated_{node_name}_seed{seed}.pt")
        ckpt_v_s = os.path.join(CKPT_DIR, ckpt_v_name)
        print(f"--- {node_name}, semilla {seed}: entrenando bajo violación ---")
        torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves / E1 / overlap sweep)
        bb_v_s, hd_v_s = run_resolvfl(tax, clients_node, cfg_s, device,
                                      ckpt_path=ckpt_v_s)
        res_v_s = eval_tree(tax, bb_v_s, hd_v_s, Xte, yte, device)

        metrics = dict(
            violated_resolvable=block_acc(res_ref_s["pred"], yte, violated_idx),
            violated_violated=block_acc(res_v_s["pred"], yte, violated_idx),
            other_resolvable=block_acc(res_ref_s["pred"], yte, other_idx),
            other_violated=block_acc(res_v_s["pred"], yte, other_idx),
            superclass_acc_resolvable=superclass_acc_from_pred(
                res_ref_s["pred"], yte, violated_idx),
            superclass_acc_violated=superclass_acc_from_pred(
                res_v_s["pred"], yte, violated_idx),
        )
        full_grid[key] = metrics
        with open(FULL_GRID_PATH, "w") as f:
            json.dump(full_grid, f, indent=2)
        print(f"  {key} completo: {metrics}")

print("\n=== Media ± desviación por nodo (3 semillas) ===")
cols = ["violated_resolvable", "violated_violated", "other_resolvable",
       "other_violated", "superclass_acc_resolvable", "superclass_acc_violated"]
print(f"{'Nodo':16s} " + " ".join(f"{c[:12]:>13s}" for c in cols))
for node_name in ALL_NODES:
    arr = {c: np.array([full_grid[f"{node_name}_seed{s}"][c] for s in ALL_SEEDS])
          for c in cols}
    row = " ".join(f"{arr[c].mean():6.4f}±{arr[c].std():5.4f}" for c in cols)
    print(f"{node_name:16s} {row}")

summary["E1_full_grid"] = full_grid
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nGuardado en Drive.")


## Controlled cohort-overlap sweep

All ten clients supervise the root. Deep-cohort sizes are 10, 8, 5, 2 and 1, giving Jaccard overlaps of 1.00, 0.80, 0.50, 0.20 and 0.10 with the root cohort. Compare Tree and ResolvFL using three seeds and 150 rounds per setting. These controlled overlaps do not describe the default Dirichlet federations.

In [ ]:
def controlled_overlap_partition(tax, X, y_leaf, K, frac_deep, rng):
    """Particion con solapamiento de cohorte CONTROLADO. Una fraccion
    frac_deep de los K clientes anota a nivel de hoja completa (supervisa
    todos los nodos internos del arbol); el resto anota solo al nivel
    inmediatamente debajo de la raiz (superclase), sin bajar mas.
    Estratificada por clase: todos los clientes reciben la MISMA
    distribucion de clases, para no confundir solapamiento con
    heterogeneidad de clase. Devuelve tambien la fraccion REAL conseguida
    (puede diferir ligeramente de la nominal por redondeo con K pequeno).

    IMPORTANTE: y_leaf contiene INDICES COMPACTOS de hoja (0..99 para
    CIFAR-100, via tax.leaf_index). y_node, en cambio, SIEMPRE debe llevar
    IDs REALES de nodo (la convencion que usa double_dirichlet_partition
    en todo el proyecto: tax.ancestor_at_depth(tax.leaves[y_leaf[i]], d)
    convierte primero con tax.leaves[...] y despues opera). Hay que
    convertir en AMBAS ramas -- deep y coarse -- no solo en una."""
    n_deep = round(frac_deep * K)
    actual_frac = n_deep / K
    clients_idx = [[] for _ in range(K)]
    for cls in np.unique(y_leaf):
        cls_idx = np.where(y_leaf == cls)[0]
        rng.shuffle(cls_idx)
        chunks = np.array_split(cls_idx, K)
        for k in range(K):
            clients_idx[k].extend(chunks[k].tolist())
    clients = []
    for k in range(K):
        idx = np.array(clients_idx[k])
        y_leaf_k = y_leaf[idx]
        real_ids_k = np.array([tax.leaves[int(yl)] for yl in y_leaf_k])
        if k < n_deep:
            y_node_k = real_ids_k  # hoja completa: el ID real de la propia hoja
        else:
            y_node_k = np.array([tax.path[rid][1] for rid in real_ids_k])  # superclase
        clients.append({"X": X[idx], "y_node": y_node_k, "y_leaf": y_leaf_k})
    return clients, actual_frac


In [ ]:
OVERLAP_LEVELS = [1.0, 0.75, 0.5, 0.25, 0.1]
OVERLAP_SEEDS = [1, 2, 3]
OVERLAP_PATH = os.path.join(CKPT_DIR, "overlap_sweep.json")

overlap_results = {}
if os.path.exists(OVERLAP_PATH):
    with open(OVERLAP_PATH) as f:
        overlap_results = json.load(f)
    print(f"Progreso previo recuperado: {list(overlap_results.keys())}")

def _save_overlap():
    with open(OVERLAP_PATH, "w") as f:
        json.dump(overlap_results, f, indent=2)

for frac_nominal in OVERLAP_LEVELS:
    level_key = f"{frac_nominal}"
    if level_key not in overlap_results:
        overlap_results[level_key] = {}
    level_dict = overlap_results[level_key]

    for seed in OVERLAP_SEEDS:
        seed_key = str(seed)
        if seed_key in level_dict and "tree" in level_dict[seed_key] and "resolvfl" in level_dict[seed_key]:
            print(f"[nivel {frac_nominal}, seed {seed}] ya completa, se salta.")
            continue

        print(f"\n=== Nivel {frac_nominal}, semilla {seed} ===")
        rng_s = np.random.default_rng(seed)
        clients_s, actual_frac = controlled_overlap_partition(tax, Xtr, ytr, K, frac_nominal, rng_s)
        cfg_s = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                          feat_dim=128, seed=seed, save_every=10)

        seed_dict = level_dict.get(seed_key, {})
        level_dict[seed_key] = seed_dict
        seed_dict["actual_frac_deep"] = actual_frac

        if "tree" not in seed_dict:
            ckpt = os.path.join(CKPT_DIR, f"overlap_tree_{frac_nominal}_{seed}.pt")
            torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves)
            bb, hd = run_fedavg_tree(tax, clients_s, cfg_s, device, ckpt_path=ckpt)
            r = eval_tree(tax, bb, hd, Xte, yte, device)
            seed_dict["tree"] = r["leaf_acc"]
            print(f"  tree: acc={r['leaf_acc']:.4f}")
            _save_overlap()

        if "resolvfl" not in seed_dict:
            ckpt = os.path.join(CKPT_DIR, f"overlap_resolvfl_{frac_nominal}_{seed}.pt")
            torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves)
            bb, hd = run_resolvfl(tax, clients_s, cfg_s, device, ckpt_path=ckpt)
            r = eval_tree(tax, bb, hd, Xte, yte, device)
            seed_dict["resolvfl"] = r["leaf_acc"]
            print(f"  resolvfl: acc={r['leaf_acc']:.4f}")
            _save_overlap()

    print(f"[nivel {frac_nominal}] completo.")

print("\nSweep completo:", list(overlap_results.keys()))


In [ ]:
# Resumen AUTONOMO: no depende de que ninguna otra celda se haya ejecutado
# antes en esta sesion -- reconstruye sus propias rutas, así que se puede
# ejecutar esta celda sola, en cualquier momento, para ver el progreso
# real sin tener que abrir ningun JSON a mano ni rerun el resto.
import os, json
import numpy as np

_CKPT_DIR = "/content/drive/MyDrive/resolvfl_cifar100"
_OVERLAP_PATH = os.path.join(_CKPT_DIR, "overlap_sweep.json")
_LEVELS_EXPECTED = [1.0, 0.75, 0.5, 0.25, 0.1]
_SEEDS_EXPECTED = [1, 2, 3]

if not os.path.exists(_OVERLAP_PATH):
    print("Todavía no hay ningún resultado guardado -- el sweep no ha empezado o Drive no está montado.")
else:
    with open(_OVERLAP_PATH) as f:
        _res = json.load(f)

    print("=== Delta_mask (ResolvFL - Tree) por nivel de solapamiento ===\n")
    rows = []
    for nominal in _LEVELS_EXPECTED:
        level_key = str(nominal)
        seeds_dict = _res.get(level_key, {})
        deltas, actual_fracs, done_seeds = [], [], []
        for seed in _SEEDS_EXPECTED:
            sd = seeds_dict.get(str(seed), {})
            if "tree" in sd and "resolvfl" in sd:
                deltas.append(sd["resolvfl"] - sd["tree"])
                actual_fracs.append(sd["actual_frac_deep"])
                done_seeds.append(seed)
        missing = [s for s in _SEEDS_EXPECTED if s not in done_seeds]
        if deltas:
            mean_d = np.mean(deltas) * 100
            std_d = np.std(deltas, ddof=1) * 100 if len(deltas) > 1 else float("nan")
            actual = np.mean(actual_fracs)
            estado = "" if not missing else f"  [incompleto -- faltan semillas {missing}]"
            print(f"solapamiento nominal={nominal:.2f} (real={actual:.2f}): "
                 f"Delta_mask = {mean_d:+.2f} +/- {std_d:.2f} puntos  (n={len(deltas)}/3 semillas){estado}")
        else:
            print(f"solapamiento nominal={nominal:.2f}: sin resultados todavía "
                 f"[faltan las {len(_SEEDS_EXPECTED)} semillas]")


## Exploratory overlap trajectories

Record training trajectories for overlaps 1.0, 0.5 and 0.1 with two seeds per setting and evaluation every 15 rounds. These exploratory runs are distinct from the three-seed main comparison.

In [ ]:
TRAJ_LEVELS = [1.0, 0.5, 0.1]
TRAJ_SEEDS = [1, 2]
TRAJ_EVAL_EVERY = 15
TRAJ_PATH = os.path.join(CKPT_DIR, "overlap_trajectory.json")

traj_results = {}
if os.path.exists(TRAJ_PATH):
    with open(TRAJ_PATH) as f:
        traj_results = json.load(f)
    print(f"Progreso previo recuperado: {list(traj_results.keys())}")

def _save_traj():
    with open(TRAJ_PATH, "w") as f:
        json.dump(traj_results, f, indent=2)

def run_with_trajectory(run_fn, tax, clients, cfg, device, ckpt_path, eval_every, Xte, yte):
    """Entrena con run_fn (run_fedavg_tree o run_resolvfl) pero evaluando
    cada eval_every rondas en vez de solo al final. Usa la MISMA logica
    interna que run_fn -- para no duplicar el bucle federado, se ejecuta
    run_fn repetidamente en tramos cortos, aprovechando su propio soporte
    de checkpoint/resume para encadenar tramos sin perder progreso."""
    trajectory = []
    total_rounds = cfg.rounds
    done = 0
    while done < total_rounds:
        next_stop = min(done + eval_every, total_rounds)
        cfg_partial = FedConfig(rounds=next_stop, local_epochs=cfg.local_epochs,
                                batch_size=cfg.batch_size, lr=cfg.lr,
                                feat_dim=cfg.feat_dim, seed=cfg.seed, save_every=eval_every)
        bb, hd = run_fn(tax, clients, cfg_partial, device, ckpt_path=ckpt_path)
        r = eval_tree(tax, bb, hd, Xte, yte, device)
        trajectory.append({"round": next_stop, "leaf_acc": r["leaf_acc"]})
        done = next_stop
    return bb, hd, trajectory


for frac_nominal in TRAJ_LEVELS:
    level_key = str(frac_nominal)
    if level_key not in traj_results:
        traj_results[level_key] = {}
    level_dict = traj_results[level_key]

    for seed in TRAJ_SEEDS:
        seed_key = str(seed)
        if seed_key in level_dict and "tree_traj" in level_dict[seed_key] and "resolvfl_traj" in level_dict[seed_key]:
            print(f"[nivel {frac_nominal}, seed {seed}] ya completa, se salta.")
            continue

        print(f"\n=== Trayectoria: nivel {frac_nominal}, semilla {seed} ===")
        rng_s = np.random.default_rng(seed)
        clients_s, actual_frac = controlled_overlap_partition(tax, Xtr, ytr, K, frac_nominal, rng_s)
        cfg_s = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                          feat_dim=128, seed=seed, save_every=TRAJ_EVAL_EVERY)

        seed_dict = level_dict.get(seed_key, {})
        level_dict[seed_key] = seed_dict
        seed_dict["actual_frac_deep"] = actual_frac

        if "tree_traj" not in seed_dict:
            ckpt = os.path.join(CKPT_DIR, f"traj_tree_{frac_nominal}_{seed}.pt")
            torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves)
            _, _, traj = run_with_trajectory(run_fedavg_tree, tax, clients_s, cfg_s, device,
                                             ckpt, TRAJ_EVAL_EVERY, Xte, yte)
            seed_dict["tree_traj"] = traj
            print(f"  tree: {[f"{p['round']}:{p['leaf_acc']:.3f}" for p in traj]}")
            _save_traj()

        if "resolvfl_traj" not in seed_dict:
            ckpt = os.path.join(CKPT_DIR, f"traj_resolvfl_{frac_nominal}_{seed}.pt")
            torch.manual_seed(seed)  # inicializacion emparejada (mismo protocolo que iNat-Aves)
            _, _, traj = run_with_trajectory(run_resolvfl, tax, clients_s, cfg_s, device,
                                             ckpt, TRAJ_EVAL_EVERY, Xte, yte)
            seed_dict["resolvfl_traj"] = traj
            print(f"  resolvfl: {[f"{p['round']}:{p['leaf_acc']:.3f}" for p in traj]}")
            _save_traj()

print("\nCompleto:", list(traj_results.keys()))


In [ ]:
# Resumen AUTONOMO: Delta_mask por ronda, para ver si la separacion es
# grande al principio y se cierra al final -- reconstruye sus propias
# rutas, no depende de celdas anteriores.
import os, json
import numpy as np

_CKPT_DIR = "/content/drive/MyDrive/resolvfl_cifar100"
_TRAJ_PATH = os.path.join(_CKPT_DIR, "overlap_trajectory.json")

if not os.path.exists(_TRAJ_PATH):
    print("Todavía no hay resultados de trayectoria guardados.")
else:
    with open(_TRAJ_PATH) as f:
        _tr = json.load(f)

    print("=== Delta_mask (ResolvFL - Tree) por ronda, cada nivel de solapamiento ===\n")
    for level_key, seeds_dict in sorted(_tr.items(), key=lambda kv: -float(kv[0])):
        print(f"--- solapamiento {level_key} ---")
        # juntar por ronda a traves de las semillas disponibles
        by_round = {}
        for seed_key, sd in seeds_dict.items():
            if "tree_traj" not in sd or "resolvfl_traj" not in sd:
                continue
            tree_by_r = {p["round"]: p["leaf_acc"] for p in sd["tree_traj"]}
            fg_by_r = {p["round"]: p["leaf_acc"] for p in sd["resolvfl_traj"]}
            for r in tree_by_r:
                if r in fg_by_r:
                    by_round.setdefault(r, []).append(fg_by_r[r] - tree_by_r[r])
        if not by_round:
            print("  sin datos completos todavía\n")
            continue
        for r in sorted(by_round):
            deltas = by_round[r]
            print(f"  ronda {r:3d}: Delta_mask = {np.mean(deltas)*100:+.2f} puntos  (n={len(deltas)} semillas)")
        print()


## 14. Seed-paired main comparison

Each of three runs uses a separately sampled double-Dirichlet partition, shared by Flat, Tree and ResolvFL. Tree and ResolvFL share architecture and initialization. Flat has a different output space, so a shared random seed does not imply identical model parameters. Partition seeds are 10001–10003 and training seeds are 20001–20003.

In [ ]:
import random

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

TABLE1_SEEDS = [1, 2, 3]
TABLE1_RESULTS_PATH = os.path.join(CKPT_DIR, "table1_multiseed_partitioned_v2.json")

REQUIRED_KEYS = {
    "resolvfl_leaf_acc", "flat_leaf_acc", "flat_abstain", "tree_leaf_acc",
    "delta_tree_flat", "delta_mask",
}

EXPECTED_PROTOCOL = {
    "K": 10, "alpha_cls": 0.5, "alpha_dep": 0.5, "rounds": 150,
    "local_epochs": 1, "batch_size": 128, "lr": 0.05, "feat_dim": 128,
    "partition_varies_by_seed": True, "seeds": TABLE1_SEEDS,
    "note": "Reemplaza a tab:cifar100-main (single-seed); particion "
            "double-Dirichlet independiente por seed, misma para los 3 metodos "
            "dentro de cada seed. FedAvg-tree y ResolvFL comparten arquitectura "
            "e inicializacion; el baseline flat tiene una cabeza distinta, asi "
            "que 'misma inicializacion' no aplica a la comparacion con el.",
}

table1_results = {"protocol": EXPECTED_PROTOCOL, "seeds": {}}
if os.path.exists(TABLE1_RESULTS_PATH):
    with open(TABLE1_RESULTS_PATH) as f:
        loaded = json.load(f)
    if loaded.get("protocol") != EXPECTED_PROTOCOL:
        raise RuntimeError(
            "El fichero existente usa un protocolo distinto al esperado. "
            "Usa un nombre de fichero nuevo o borra el anterior antes de seguir.")
    table1_results = loaded
table1_results.setdefault("seeds", {})

for seed in TABLE1_SEEDS:
    seed_entry = table1_results["seeds"].get(str(seed), {})
    if REQUIRED_KEYS.issubset(seed_entry):
        print(f"[ya hecho y completo] semilla {seed}")
        continue

    partition_seed = 10_000 + seed
    training_seed = 20_000 + seed
    print(f"\n===== SEED {seed} | partition={partition_seed}, training={training_seed} =====")

    rng_s = np.random.default_rng(partition_seed)
    clients_s = double_dirichlet_partition(tax, Xtr, ytr, K,
                                           alpha_cls=0.5, alpha_dep=0.5, rng=rng_s)
    # clients_s se reutiliza sin copiar entre los 3 metodos: verificado que
    # run_resolvfl / run_flat_fedavg / run_fedavg_tree solo LEEN de cada
    # cliente (torch.tensor(c["X"]/c["y_node"]) crea tensores nuevos cada
    # vez); no hace falta copy.deepcopy.

    cfg_s = FedConfig(rounds=150, local_epochs=1, batch_size=128, lr=0.05,
                      feat_dim=128, seed=training_seed, save_every=5)

    ckpt_fg = os.path.join(CKPT_DIR, f"table1_v2_seed{seed}_resolvfl.pt")
    ckpt_fl = os.path.join(CKPT_DIR, f"table1_v2_seed{seed}_flat.pt")
    ckpt_tr = os.path.join(CKPT_DIR, f"table1_v2_seed{seed}_tree.pt")

    print("--- ResolvFL (masked) ---")
    set_all_seeds(training_seed)
    bb_fg, hd_fg = run_resolvfl(tax, clients_s, cfg_s, device, ckpt_path=ckpt_fg)
    res_fg = eval_tree(tax, bb_fg, hd_fg, Xte, yte, device)

    print("--- FedAvg-flat ---")
    set_all_seeds(training_seed)
    bb_fl, hd_fl = run_flat_fedavg(tax, clients_s, cfg_s, device, ckpt_path=ckpt_fl)
    res_fl = eval_flat(tax, bb_fl, hd_fl, Xte, yte, device)

    print("--- FedAvg-tree (unmasked) ---")
    set_all_seeds(training_seed)
    bb_tr, hd_tr = run_fedavg_tree(tax, clients_s, cfg_s, device, ckpt_path=ckpt_tr)
    res_tr = eval_tree(tax, bb_tr, hd_tr, Xte, yte, device)

    table1_results["seeds"][str(seed)] = {
        "partition_seed": int(partition_seed),
        "training_seed": int(training_seed),
        "flat_leaf_acc": float(res_fl["leaf_acc"]),
        "flat_abstain": float(res_fl["abstain_rate"]),
        "tree_leaf_acc": float(res_tr["leaf_acc"]),
        "resolvfl_leaf_acc": float(res_fg["leaf_acc"]),
        "delta_tree_flat": float(res_tr["leaf_acc"] - res_fl["leaf_acc"]),
        "delta_mask": float(res_fg["leaf_acc"] - res_tr["leaf_acc"]),
    }
    with open(TABLE1_RESULTS_PATH, "w") as f:
        json.dump(table1_results, f, indent=2)
    print(f"semilla {seed} guardada: {table1_results['seeds'][str(seed)]}")

print("\n=== Tabla I multi-semilla (particion + entrenamiento por seed): media +/- std ===")
for key in ["flat_leaf_acc", "flat_abstain", "tree_leaf_acc", "resolvfl_leaf_acc",
           "delta_tree_flat", "delta_mask"]:
    vals = [table1_results["seeds"][str(s)][key] for s in TABLE1_SEEDS]
    m = np.mean(vals)
    sd = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
    print(f"  {key}: {m:.2f} +/- {sd:.2f}  (por semilla: {[round(v,2) for v in vals]})")
